# Modelling experiment: features → models → training → evaluation

Copy this notebook into your personal folder. Change the feature functions, candidates, and parameters here; shared modules handle the experiment mechanics.

**Workflow:** 80% development data + 20% untouched holdout → compare candidates on the same five stratified folds → choose using CV → explicitly open the holdout once → fit on all labelled battles and generate a Kaggle submission.

Every candidate gets K-fold validation. Imputation, constant removal, optional feature selection, and scaling are fitted within each fold. Test data is read only when generating a submission. The final full-data fit is separate from evaluation.

The default run compares team-only features, team + timeline features, automatic feature selection, and a random forest. This is a starting process, not a tuned solution.

In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/pokemon_challenge").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open the repository folder or run from a notebook inside it.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from pokemon_challenge.experiment import BattleExperiment, Candidate
from pokemon_challenge.features.basic import team_stats, timeline_summary

OWNER = "your_name"  # change to your folder name in your own copy
RUN = datetime.now().strftime("%Y%m%d_%H%M%S_baseline")
print("Run:", OWNER, RUN)


## 1. Choose or define feature groups

Each function takes **one battle** and returns a dictionary of numeric values, with the same keys for every battle. `battle_id` and `player_won` are removed before calling it. Return `None` or `float('nan')` for missing values; the fold-fitted imputer handles them.

Functions must be stateless: do not fit encoders, scalers, selectors, or statistics across the dataset here. Learned preprocessing belongs in the sklearn pipeline. The basic runner supports numeric features; categorical strings need an explicitly added fold-fitted encoder, not arbitrary numbering.

You can define a function directly in this notebook, add it to `FEATURE_FUNCTIONS`, and reference its group name in a candidate. Move it into `src/pokemon_challenge/features/` when ready to share.

In [ ]:
FEATURE_FUNCTIONS = {
    "team": team_stats,
    "timeline": timeline_summary,
}

# Example of a custom function defined entirely in this notebook:
# def my_features(battle):
#     return {"p1_leftovers_count": sum(
#         p.get("item") == "leftovers" for p in battle.get("p1_team_details", []))}
# FEATURE_FUNCTIONS["items"] = my_features


## 2. Load features and create fixed validation assignments

Settings come from `configs/validation.json`: accuracy, seed 42, five stratified folds, and a 20% holdout. The runner saves every battle's partition/fold assignment and its hash. Experiments using the same training file, order, and validation settings use identical assignments.

Battle is the split unit. The current splitter does not group related or duplicate battles; inspect that assumption before relying on the scores. Do not change the holdout seed to search for a better result.

A new run folder is required for each experiment. Rerun the first cell to create a new run name before constructing another experiment. Saving custom function source is best-effort; retain this notebook alongside the exact implementation commit.

In [ ]:
experiment = BattleExperiment(
    owner=OWNER,
    run=RUN,
    feature_functions=FEATURE_FUNCTIONS,
    # Optional overrides; use the team's agreed settings when comparing results:
    # validation={"seed": 42, "n_splits": 5, "holdout_size": 0.2},
)
display(experiment.X.head())
print("Numeric input features:", experiment.X.shape[1])
display(experiment.assignments.groupby(["partition", "validation_fold"]).size().rename("battles").to_frame())


## 3. Define feature and model candidates

- Change `groups` to compare feature blocks.
- Change `model` to `logistic_regression`, `random_forest`, or `hist_gradient_boosting`.
- Set `percentile=50` to retain the top half of nonconstant features using a fold-fitted ANOVA selector; `100` keeps them all. This is one basic selection strategy, not proof of importance or causality.
- Change model settings through `params`. Each named setting combination is another candidate.

Automatic selection may choose different variables in different folds. The runner saves their selection frequencies. All candidates first remove constant columns using development-fold training rows only.

In [ ]:
candidates = [
    Candidate("team_logistic", groups=("team",)),
    Candidate("team_timeline_logistic", groups=("team", "timeline")),
    Candidate("selected_logistic", groups=("team", "timeline"), percentile=50),
    Candidate("team_timeline_forest", groups=("team", "timeline"), model="random_forest",
              params={"n_estimators": 100, "max_depth": 8, "min_samples_leaf": 5}),
]
results = experiment.compare(candidates)
display(results)
print("Highest mean CV accuracy:", experiment.best_candidate)


## 4. Inspect selection and compare more configurations (optional)

`cv_accuracy` is the mean of the five validation accuracies; `fold_std` describes variation across those folds, not a confidence interval. `dummy_accuracy` is the majority-class reference evaluated on the same folds. Examine per-fold scores and runtime, not just the highest mean.

To test more settings, call `compare` with **new unique candidate names**. It appends results using the existing splits. This is manual model/hyperparameter selection driven by CV; the runner does not silently perform a large search.

In [ ]:
selection_files = sorted(experiment.output_path.glob("selection_frequency_*.csv"))
for path in selection_files:
    print(path.name)
    display(pd.read_csv(path).sort_values("selected_in_folds", ascending=False).head(15))

# Add candidates before looking at holdout results:
# results = experiment.compare([
#     Candidate("logistic_C_01", groups=("team", "timeline"), params={"C": 0.1}),
#     Candidate("hist_boosting", groups=("team", "timeline"), model="hist_gradient_boosting",
#               params={"max_iter": 100, "max_leaf_nodes": 15}),
# ])
# display(results)


## 5. Final holdout evaluation — explicit decision

Leave this **False** while choosing features and models. Set it to **True** only when you have chosen your approach from CV. The chosen pipeline is fitted on the 80% development subset and evaluated on the untouched 20% holdout.

Opening the holdout locks further comparisons in this experiment. Do not tune on its score or repeatedly create new runs to work around the lock; doing so would turn the holdout into another tuning set. The guard is per Python experiment object, not a cross-team access control system.

In [ ]:
EVALUATE_HOLDOUT = False
CHOSEN_CANDIDATE = experiment.best_candidate  # or a specific compared candidate name

if EVALUATE_HOLDOUT:
    holdout = experiment.evaluate_holdout(CHOSEN_CANDIDATE)
    print("Candidate:", holdout["candidate"])
    print("Holdout accuracy:", holdout["accuracy"])
    display(pd.DataFrame(holdout["confusion_matrix"],
                         index=["actual loss", "actual win"], columns=["predicted loss", "predicted win"]))
    display(pd.DataFrame(holdout["classification_report"]).T)
    print("Selected features:", holdout["selected_features"])
else:
    print("Holdout remains unopened. Continue comparing using CV only.")


## 6. Train on all labels and generate a submission (optional)

After final evaluation, set the flag below to **True**. The runner refits the locked candidate on all 10,000 labelled battles, including the former holdout, and predicts official test battles. This final fit has no new unbiased local evaluation score; the earlier CV and holdout results describe the chosen method.

The sample submission controls ID association and row order. Overlapping train/test ID ranges are allowed. No upload is automatic. The saved model accepts the selected numeric feature columns; reproducing raw-battle predictions also requires the feature functions and exact configuration.

In [ ]:
MAKE_SUBMISSION = False
if MAKE_SUBMISSION:
    submission = experiment.make_submission()
    display(submission.head())
    print("Upload manually:", experiment.output_path / "submission.csv")
else:
    print("Submission generation skipped.")


## 7. Artifacts and experiment notes

The unique ignored run folder contains:

- `experiment.json`: validation settings, data/split hashes, versions, code commit and dirty-tree status.
- `feature_sources.json`: feature function source where available.
- `splits.csv`: saved holdout and development-fold assignments.
- `candidates.json`, `cv_results.csv`, and `selection_frequency_*.csv`: configurations and CV evidence.
- After holdout evaluation: metrics, selected features, and battle-level predictions.
- After final fitting: `model.joblib`, `submission.csv`, and `final_fit.json`.

Record the hypothesis, interpretation, and exact run/commit in your notes. Uncommitted changes are flagged in the manifest; commit the implementation and retain this notebook/configuration before relying on a final result. Saved source snippets help auditing but do not automatically package custom imports or globals.

Only load model files you trust. Saved outputs are local and ignored by Git; commit the notebook/shared code separately and preserve final artifacts with your team.

In [ ]:
print("Run folder:", experiment.output_path)
for path in sorted(experiment.output_path.iterdir()):
    print(" -", path.name)
